# Weeks 2–3 – Dataset Structuring, Validation & Mortgage Rate Enrichment

Run `data_loading.ipynb` first — this notebook loads the CSVs it produces.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Load Residential-filtered datasets produced by data_loading.ipynb
listings_res = pd.read_csv('listings_residential.csv', low_memory=False)
sold_res     = pd.read_csv('sold_residential.csv',     low_memory=False)

print(f"Loaded listings_residential.csv : {len(listings_res):,} rows x {listings_res.shape[1]} cols")
print(f"Loaded sold_residential.csv     : {len(sold_res):,} rows x {sold_res.shape[1]} cols")

## Dataset Understanding

In [ ]:
print("=== LISTINGS DATASET ===")
print(f"Shape: {listings_res.shape[0]:,} rows x {listings_res.shape[1]} columns")
print("\nColumn dtypes:")
print(listings_res.dtypes.to_string())

print("\n=== SOLD DATASET ===")
print(f"Shape: {sold_res.shape[0]:,} rows x {sold_res.shape[1]} columns")
print("\nColumn dtypes:")
print(sold_res.dtypes.to_string())

In [ ]:
# Unique PropertyType values (confirm Residential-only filter held after CSV round-trip)
print("Listings PropertyType values:")
print(listings_res['PropertyType'].value_counts(dropna=False))

print("\nSold PropertyType values:")
print(sold_res['PropertyType'].value_counts(dropna=False))

## Missing Value Analysis

In [ ]:
def missing_report(df, name):
    total = len(df)
    missing = df.isnull().sum()
    pct = (missing / total * 100).round(2)
    report = pd.DataFrame({'missing_count': missing, 'missing_pct': pct})
    report = report[report['missing_count'] > 0].sort_values('missing_pct', ascending=False)
    print(f"\n=== {name} – Missing Value Report ({total:,} rows) ===")
    print(report.to_string())
    high_missing = report[report['missing_pct'] > 90]
    print(f"\nColumns with >90% missing ({len(high_missing)}):")
    print(high_missing.to_string() if len(high_missing) else "  None")
    return report

listings_missing = missing_report(listings_res, "LISTINGS (Residential)")
sold_missing     = missing_report(sold_res,     "SOLD (Residential)")

## Numeric Distribution Review

In [ ]:
numeric_fields = [
    'ClosePrice', 'ListPrice', 'OriginalListPrice',
    'LivingArea', 'LotSizeAcres', 'BedroomsTotal',
    'BathroomsTotalInteger', 'DaysOnMarket', 'YearBuilt'
]
numeric_fields = [f for f in numeric_fields if f in sold_res.columns]

# Percentile summary table
percentiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
summary = sold_res[numeric_fields].describe(percentiles=percentiles)
print("=== SOLD (Residential) – Numeric Distribution Summary ===")
print(summary.to_string())

# Detailed stats for the three deliverable fields
for col in ['ClosePrice', 'LivingArea', 'DaysOnMarket']:
    if col in sold_res.columns:
        s = sold_res[col].dropna()
        print(f"\n--- {col} ---")
        print(f"  count:  {s.count():,}")
        print(f"  mean:   {s.mean():,.2f}")
        print(f"  median: {s.median():,.2f}")
        print(f"  min:    {s.min():,.2f}")
        print(f"  max:    {s.max():,.2f}")
        print(f"  p25:    {s.quantile(0.25):,.2f}")
        print(f"  p75:    {s.quantile(0.75):,.2f}")

In [ ]:
# Histograms (capped at 99th percentile for readability)
n_cols = 3
n_rows = int(np.ceil(len(numeric_fields) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_fields):
    data = sold_res[col].dropna()
    cap  = data.quantile(0.99)
    data_capped = data[data <= cap]
    axes[i].hist(data_capped, bins=50, color='steelblue', edgecolor='white', linewidth=0.3)
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')
    axes[i].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    axes[i].tick_params(axis='x', rotation=30)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Sold (Residential) – Numeric Field Distributions (capped at 99th pct)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots (capped at 99th percentile for readability)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_fields):
    data = sold_res[col].dropna()
    cap  = data.quantile(0.99)
    data_capped = data[data <= cap]
    axes[i].boxplot(data_capped, vert=True, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6))
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_ylabel('Value')
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Sold (Residential) – Boxplots (capped at 99th pct)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Suggested Intern Questions (EDA Answers)

In [ ]:
# Q1 – Residential vs. other property type share
# (loading full combined sold to show full share — reload raw file)
sold_all = pd.read_csv('sold_residential.csv', low_memory=False)
print("=== Q1: PropertyType share (all sold records) ===")
print("Note: loaded dataset is already Residential-filtered; see data_loading.ipynb for full breakdown.")
print(sold_all['PropertyType'].value_counts(dropna=False))

# Q2 – Median and average close prices
print(f"\n=== Q2: ClosePrice stats (Residential Sold) ===")
print(f"  Median:  ${sold_res['ClosePrice'].median():,.0f}")
print(f"  Average: ${sold_res['ClosePrice'].mean():,.0f}")

# Q3 – Days on Market distribution
print(f"\n=== Q3: DaysOnMarket distribution ===")
dom = sold_res['DaysOnMarket'].dropna()
print(f"  Median: {dom.median():.0f} days")
print(f"  Mean:   {dom.mean():.1f} days")
print(f"  p25:    {dom.quantile(0.25):.0f} days")
print(f"  p75:    {dom.quantile(0.75):.0f} days")
print(f"  p90:    {dom.quantile(0.90):.0f} days")

# Q4 – % sold above vs. below list price
print(f"\n=== Q4: Sold above vs. below OriginalListPrice ===")
valid = sold_res[['ClosePrice', 'OriginalListPrice']].dropna()
above = (valid['ClosePrice'] >= valid['OriginalListPrice']).sum()
below = (valid['ClosePrice'] <  valid['OriginalListPrice']).sum()
print(f"  At or above list: {above:,}  ({above/len(valid)*100:.1f}%)")
print(f"  Below list:       {below:,}  ({below/len(valid)*100:.1f}%)")

# Q5 – Date consistency issues (close before listing)
close_dt   = pd.to_datetime(sold_res['CloseDate'],           errors='coerce')
listing_dt = pd.to_datetime(sold_res['ListingContractDate'], errors='coerce')
bad_dates  = (close_dt < listing_dt).sum()
print(f"\n=== Q5: CloseDate before ListingContractDate ===")
print(f"  Flagged records: {bad_dates:,}")

# Q6 – Counties with highest median close price
print(f"\n=== Q6: Top 10 counties by median ClosePrice ===")
county_col = 'CountyOrParish' if 'CountyOrParish' in sold_res.columns else None
if county_col:
    county_median = (
        sold_res.groupby(county_col)['ClosePrice']
        .median()
        .sort_values(ascending=False)
        .head(10)
    )
    print(county_median.apply(lambda x: f"${x:,.0f}").to_string())
else:
    print("  CountyOrParish column not found")

## Mortgage Rate Enrichment (FRED MORTGAGE30US)

In [ ]:
# Step 1 – Fetch the 30-year fixed mortgage rate series from FRED
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

# FRED uses '.' as a placeholder for unreleased future dates — drop those rows
mortgage = mortgage[pd.to_numeric(mortgage['rate_30yr_fixed'], errors='coerce').notna()].copy()
mortgage['rate_30yr_fixed'] = mortgage['rate_30yr_fixed'].astype(float)

print(f"FRED data fetched: {len(mortgage):,} weekly observations")
print(f"Date range: {mortgage['date'].min().date()} → {mortgage['date'].max().date()}")
print(mortgage.tail())

# Step 2 – Resample weekly → monthly average
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
    mortgage.groupby('year_month')['rate_30yr_fixed']
    .mean()
    .reset_index()
)
print(f"\nMonthly averages: {len(mortgage_monthly)} months")
print(mortgage_monthly.tail())

In [ ]:
# Step 3 – Create year_month join keys on the MLS datasets
sold_res['year_month']     = pd.to_datetime(sold_res['CloseDate'],              errors='coerce').dt.to_period('M')
listings_res['year_month'] = pd.to_datetime(listings_res['ListingContractDate'], errors='coerce').dt.to_period('M')

# Step 4 – Left-merge mortgage rates onto both datasets
sold_with_rates     = sold_res.merge(mortgage_monthly,     on='year_month', how='left')
listings_with_rates = listings_res.merge(mortgage_monthly, on='year_month', how='left')

# Step 5 – Validate: no nulls expected for months within the FRED series range
sold_null_rates     = sold_with_rates['rate_30yr_fixed'].isnull().sum()
listings_null_rates = listings_with_rates['rate_30yr_fixed'].isnull().sum()

print("=== Merge Validation ===")
print(f"Sold     – null rate_30yr_fixed: {sold_null_rates}")
print(f"Listings – null rate_30yr_fixed: {listings_null_rates}")

print("\nSold sample (CloseDate / year_month / ClosePrice / rate):")
print(
    sold_with_rates[['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']]
    .dropna(subset=['rate_30yr_fixed'])
    .head()
    .to_string()
)

## Save Output CSVs

In [ ]:
# Convert Period to string before writing (Period is not CSV-serialisable)
sold_with_rates['year_month']     = sold_with_rates['year_month'].astype(str)
listings_with_rates['year_month'] = listings_with_rates['year_month'].astype(str)

sold_with_rates.to_csv('sold_residential_with_rates.csv',         index=False)
listings_with_rates.to_csv('listings_residential_with_rates.csv', index=False)

print("Saved:")
print(f"  sold_residential_with_rates.csv       – {len(sold_with_rates):,} rows")
print(f"  listings_residential_with_rates.csv   – {len(listings_with_rates):,} rows")